# Test-7 -- From-Scratch Transformer, English to Odia
Encoder-decoder transformer implemented per assignment section 5.6
(d_model=128, heads=4, N=2 encoder/decoder blocks). This notebook reconstructs
the exact source files developed and unit-tested locally, then runs the real
training pass on GPU.


In [ ]:
!pip install -q datasets tokenizers sacrebleu


In [ ]:
import base64
from pathlib import Path

REPO_ROOT = Path("/kaggle/working")
_FILES_B64 = {"configs/base.py": "IiIiRnJvemVuIGh5cGVycGFyYW1ldGVycyBhbmQgcGF0aHMgZm9yIHRoZSBUZXN0LTcgRW4tPk9kaWEgdHJhbnNmb3JtZXIuCgpOdW1iZXJzIGJlbG93IGFyZSBtZWFzdXJlZCwgbm90IGFzc3VtZWQgLS0gc2VlIHRoZSBNQVhfTEVOIHJldGVudGlvbgptZWFzdXJlbWVudCBhZ2FpbnN0IGEgcmVhbCBTYW1hbmFudGFyIHNhbXBsZSAoRW5nbGlzaCBtZWFuL21lZGlhbiAxNy41LzEzCnN1YndvcmRzLCBPZGlhIG1lYW4vbWVkaWFuIDQ5LjgvMzkgc3Vid29yZHMsIDc4LjAlIHBhaXItcmV0ZW50aW9uIGF0Ck1BWF9MRU49NjQpIGJlZm9yZSBjaGFuZ2luZyBTVUJTRVRfU0laRSBvciBNQVhfTEVOLgoiIiIKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUkVQT19ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0KREFUQV9SQVdfRElSID0gUkVQT19ST09UIC8gImRhdGEiIC8gInJhdyIKREFUQV9QUk9DRVNTRURfRElSID0gUkVQT19ST09UIC8gImRhdGEiIC8gInByb2Nlc3NlZCIKVE9LRU5JWkVSX0RJUiA9IFJFUE9fUk9PVCAvICJ0b2tlbml6ZXJzIgpDSEVDS1BPSU5UX0RJUiA9IFJFUE9fUk9PVCAvICJjaGVja3BvaW50cyIKUkVQT1JUU19ESVIgPSBSRVBPX1JPT1QgLyAicmVwb3J0cyIKCiMgLS0tIERhdGFzZXQgLS0tCkhGX0RBVEFTRVRfSUQgPSAiYWk0YmhhcmF0L3NhbWFuYW50YXIiCkhGX0RBVEFTRVRfQ09ORklHID0gIm9yIgpSQU5ET01fU0VFRCA9IDQyCgojIFdvcmQtY291bnQgZmlsdGVyIGFwcGxpZWQgYmVmb3JlIHRva2VuaXphdGlvbiAoY2hlYXAgZmlyc3QgcGFzcykuCk1JTl9XT1JEUyA9IDMKTUFYX1dPUkRTID0gNjAKCiMgRmluYWwgdGFyZ2V0IGNvdW50cyAocG9zdCBNQVhfTEVOIGZpbHRlcmluZykuClRSQUlOX1NJWkUgPSAzNl8wMDAKVkFMX1NJWkUgPSAyXzAwMApURVNUX1NJWkUgPSAyXzAwMApUT1RBTF9TSVpFID0gVFJBSU5fU0laRSArIFZBTF9TSVpFICsgVEVTVF9TSVpFICAjIDQwLDAwMAoKIyBPdmVyLWNvbGxlY3Rpb24gdGFyZ2V0IGJlZm9yZSB0aGUgTUFYX0xFTiBmaWx0ZXIsIHRvIG5ldCBUT1RBTF9TSVpFCiMgcGFpcnMgYWZ0ZXIgfjc4JSBtZWFzdXJlZCByZXRlbnRpb24gYXQgTUFYX0xFTj02NC4KQ0FORElEQVRFX1BPT0xfU0laRSA9IDU4XzAwMAoKIyAtLS0gVG9rZW5pemF0aW9uIC0tLQpFTl9WT0NBQl9TSVpFID0gOF8wMDAKT1JfVk9DQUJfU0laRSA9IDhfMDAwClNQRUNJQUxfVE9LRU5TID0gWyI8UEFEPiIsICI8U09TPiIsICI8RU9TPiIsICI8VU5LPiJdClBBRF9JRCwgU09TX0lELCBFT1NfSUQsIFVOS19JRCA9IDAsIDEsIDIsIDMKTUFYX0xFTiA9IDY0ICAjIHN1YndvcmQgdG9rZW5zLCBpbmNsdWRpbmcgPFNPUz4vPEVPUz4KCiMgLS0tIE1vZGVsIChpbXBsZW1lbnRzIGFzc2lnbm1lbnQgc2VjdGlvbiA1LjYpIC0tLQpEX01PREVMID0gMTI4Ck5fSEVBRFMgPSA0CkRfRkYgPSA1MTIKTl9FTkNPREVSX0xBWUVSUyA9IDIKTl9ERUNPREVSX0xBWUVSUyA9IDIKRFJPUE9VVCA9IDAuMQpUSUVfT1VUUFVUX1BST0pFQ1RJT04gPSBGYWxzZSAgIyBhc3NpZ25tZW50IHNwZWNpZmllcyAibGluZWFyK3NvZnRtYXgiOyB0eWluZyBpcyBhbiBvcHRpb25hbCBleHRyYQoKIyAtLS0gVHJhaW5pbmcgLS0tCkJBVENIX1NJWkVfS0FHR0xFID0gMTI4CkJBVENIX1NJWkVfTE9DQUxfU01PS0UgPSA4Ck5VTV9FUE9DSFNfS0FHR0xFID0gMTgKTlVNX0VQT0NIU19MT0NBTF9TTU9LRSA9IDMKQURBTV9CRVRBUyA9ICgwLjksIDAuOTgpCkFEQU1fRVBTID0gMWUtOQpXQVJNVVBfU1RFUFMgPSA5MDAKR1JBRF9DTElQX05PUk0gPSAxLjAKCiMgLS0tIEluZmVyZW5jZSAtLS0KR1JFRURZX01BWF9ERUNPREVfTEVOID0gTUFYX0xFTgpCRUFNX1dJRFRIID0gNApCRUFNX0xFTkdUSF9QRU5BTFRZID0gMC42CgojIC0tLSBFdmFsdWF0aW9uIC0tLQpOVU1fU0FNUExFX1RSQU5TTEFUSU9OUyA9IDUKTE9OR19TRU5URU5DRV9QRVJDRU5USUxFID0gMC45MAo=", "src/data/clean.py": "aW1wb3J0IHJlCmltcG9ydCB1bmljb2RlZGF0YQoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0IE1BWF9XT1JEUywgTUlOX1dPUkRTCgpfWldTUCA9ICLigIsiCl9CT00gPSAi77u/IgpfWldKID0gIuKAjSIKX1pXTkogPSAi4oCMIgpfWldfSk9JTkVSUyA9IF9aV0ogKyBfWldOSgoKX0VER0VfSk9JTkVSX1JVTiA9IHJlLmNvbXBpbGUoZiJeW3tfWldfSk9JTkVSU31dK3xbe19aV19KT0lORVJTfV0rJCIpCl9JTlRFUklPUl9KT0lORVJfUlVOID0gcmUuY29tcGlsZShmIlt7X1pXX0pPSU5FUlN9XXt7Mix9fSIpCl9XSElURVNQQUNFX1JVTiA9IHJlLmNvbXBpbGUociJccysiKQoKCmRlZiBuZmNfbm9ybWFsaXplKHRleHQ6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZDIiwgdGV4dCkKCgpkZWYgc3RyaXBfemVyb193aWR0aCh0ZXh0OiBzdHIpIC0+IHN0cjoKICAgIHRleHQgPSB0ZXh0LnJlcGxhY2UoX1pXU1AsICIiKS5yZXBsYWNlKF9CT00sICIiKQogICAgdGV4dCA9IF9FREdFX0pPSU5FUl9SVU4uc3ViKCIiLCB0ZXh0KQogICAgIyBhIHJ1biBvZiAyKyBaV0ovWldOSiBpbiB0aGUgaW50ZXJpb3IgaXMgYSBzY3JhcGluZyBhcnRpZmFjdDsgYSBsb25lCiAgICAjIG9uZSBpcyBtZWFuaW5nZnVsIEluZGljIGNvbmp1bmN0LWZvcm1hdGlvbiBhbmQgbXVzdCBzdXJ2aXZlCiAgICB0ZXh0ID0gX0lOVEVSSU9SX0pPSU5FUl9SVU4uc3ViKGxhbWJkYSBtOiBtLmdyb3VwKDApWzBdLCB0ZXh0KQogICAgcmV0dXJuIHRleHQKCgpkZWYgbm9ybWFsaXplX3doaXRlc3BhY2UodGV4dDogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gX1dISVRFU1BBQ0VfUlVOLnN1YigiICIsIHRleHQpLnN0cmlwKCkKCgpkZWYgY2xlYW5fdGV4dCh0ZXh0OiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBub3JtYWxpemVfd2hpdGVzcGFjZShzdHJpcF96ZXJvX3dpZHRoKG5mY19ub3JtYWxpemUodGV4dCkpKQoKCmRlZiBwYXNzZXNfd29yZF9jb3VudCh0ZXh0OiBzdHIsIG1pbl93b3JkczogaW50ID0gTUlOX1dPUkRTLCBtYXhfd29yZHM6IGludCA9IE1BWF9XT1JEUykgLT4gYm9vbDoKICAgIG4gPSBsZW4odGV4dC5zcGxpdCgpKQogICAgcmV0dXJuIG1pbl93b3JkcyA8PSBuIDw9IG1heF93b3JkcwoKCmRlZiBpbnNwZWN0X25mY19hbm9tYWxpZXModGV4dHM6IGxpc3Rbc3RyXSwgc2FtcGxlX3NpemU6IGludCA9IDUwMCkgLT4gZGljdDoKICAgIHNhbXBsZSA9IHRleHRzWzpzYW1wbGVfc2l6ZV0KICAgIGNoYW5nZWQgPSBbdCBmb3IgdCBpbiBzYW1wbGUgaWYgdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCB0KSAhPSB0XQogICAgbm9uX2lkZW1wb3RlbnQgPSBbCiAgICAgICAgdCBmb3IgdCBpbiBzYW1wbGUKICAgICAgICBpZiB1bmljb2RlZGF0YS5ub3JtYWxpemUoIk5GQyIsIHQpICE9IHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZDIiwgdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCB0KSkKICAgIF0KICAgIHJldHVybiB7CiAgICAgICAgInNhbXBsZV9zaXplIjogbGVuKHNhbXBsZSksCiAgICAgICAgImNoYW5nZWRfYnlfbmZjIjogbGVuKGNoYW5nZWQpLAogICAgICAgICJjaGFuZ2VkX2ZyYWN0aW9uIjogbGVuKGNoYW5nZWQpIC8gbGVuKHNhbXBsZSkgaWYgc2FtcGxlIGVsc2UgMC4wLAogICAgICAgICJub25faWRlbXBvdGVudF9jb3VudCI6IGxlbihub25faWRlbXBvdGVudCksCiAgICB9Cg==", "src/data/download.py": "aW1wb3J0IHN5cwoKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIGRhdGFzZXRzIGltcG9ydCBsb2FkX2RhdGFzZXQKCmZyb20gY29uZmlncy5iYXNlIGltcG9ydCAoCiAgICBDQU5ESURBVEVfUE9PTF9TSVpFLAogICAgREFUQV9SQVdfRElSLAogICAgSEZfREFUQVNFVF9DT05GSUcsCiAgICBIRl9EQVRBU0VUX0lELAopCmZyb20gc3JjLmRhdGEuY2xlYW4gaW1wb3J0IGNsZWFuX3RleHQsIHBhc3Nlc193b3JkX2NvdW50CgojIHBhcnF1ZXQgb3ZlciB0c3Y6IGNhbmRpZGF0ZS9wcm9jZXNzZWQgdGV4dCBjYW4gY29udGFpbiBhcmJpdHJhcnkgcHVuY3R1YXRpb24KIyAoaW5jbHVkaW5nIGxpdGVyYWwgdGFiIG9yIG5ld2xpbmUgYnl0ZXMgZnJvbSBzY3JhcGVkIHdlYiB0ZXh0KSwgd2hpY2ggYQojIHRhYi1zZXBhcmF0ZWQgZm9ybWF0IHdvdWxkIG5lZWQgdG8gZXNjYXBlOyBwYXJxdWV0IHJvdW5kLXRyaXBzIGl0IGV4YWN0bHkKQ0FORElEQVRFU19QQVRIID0gREFUQV9SQVdfRElSIC8gImNhbmRpZGF0ZXMucGFycXVldCIKCgpkZWYgY29sbGVjdF9jYW5kaWRhdGVzKHBvb2xfc2l6ZTogaW50ID0gQ0FORElEQVRFX1BPT0xfU0laRSkgLT4gbGlzdFtkaWN0XToKICAgIHN0cmVhbSA9IGxvYWRfZGF0YXNldChIRl9EQVRBU0VUX0lELCBIRl9EQVRBU0VUX0NPTkZJRywgc3BsaXQ9InRyYWluIiwgc3RyZWFtaW5nPVRydWUpCgogICAgc2Vlbl9zcmMgPSBzZXQoKQogICAgY2FuZGlkYXRlcyA9IFtdCiAgICBleGhhdXN0ZWQgPSBUcnVlCiAgICBmb3IgZXhhbXBsZSBpbiBzdHJlYW06CiAgICAgICAgc3JjID0gY2xlYW5fdGV4dChleGFtcGxlWyJzcmMiXSkKICAgICAgICB0Z3QgPSBjbGVhbl90ZXh0KGV4YW1wbGVbInRndCJdKQoKICAgICAgICBpZiBub3Qgc3JjIG9yIG5vdCB0Z3Q6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgc3JjIGluIHNlZW5fc3JjOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG5vdCAocGFzc2VzX3dvcmRfY291bnQoc3JjKSBhbmQgcGFzc2VzX3dvcmRfY291bnQodGd0KSk6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHNlZW5fc3JjLmFkZChzcmMpCiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoeyJzcmMiOiBzcmMsICJ0Z3QiOiB0Z3R9KQoKICAgICAgICBpZiBsZW4oY2FuZGlkYXRlcykgPj0gcG9vbF9zaXplOgogICAgICAgICAgICBleGhhdXN0ZWQgPSBGYWxzZQogICAgICAgICAgICBicmVhawoKICAgIGlmIGV4aGF1c3RlZDoKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJXQVJOSU5HOiBzdHJlYW0gZXhoYXVzdGVkIGJlZm9yZSByZWFjaGluZyBDQU5ESURBVEVfUE9PTF9TSVpFPSIKICAgICAgICAgICAgZiJ7cG9vbF9zaXplfTsgY29sbGVjdGVkIG9ubHkge2xlbihjYW5kaWRhdGVzKX0gY2FuZGlkYXRlcy4iLAogICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgKQoKICAgIHJldHVybiBjYW5kaWRhdGVzCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgREFUQV9SQVdfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpZiBDQU5ESURBVEVTX1BBVEguZXhpc3RzKCk6CiAgICAgICAgZXhpc3RpbmcgPSBwZC5yZWFkX3BhcnF1ZXQoQ0FORElEQVRFU19QQVRIKQogICAgICAgIHByaW50KGYie0NBTkRJREFURVNfUEFUSH0gYWxyZWFkeSBleGlzdHMgd2l0aCB7bGVuKGV4aXN0aW5nKX0gY2FuZGlkYXRlczsgc2tpcHBpbmcgZG93bmxvYWQuIikKICAgICAgICByZXR1cm4KCiAgICBjYW5kaWRhdGVzID0gY29sbGVjdF9jYW5kaWRhdGVzKCkKCiAgICBkZiA9IHBkLkRhdGFGcmFtZShjYW5kaWRhdGVzLCBjb2x1bW5zPVsic3JjIiwgInRndCJdKQogICAgZGYudG9fcGFycXVldChDQU5ESURBVEVTX1BBVEgsIGluZGV4PUZhbHNlKQogICAgcHJpbnQoZiJzYXZlZCB7bGVuKGRmKX0gY2FuZGlkYXRlcyB0byB7Q0FORElEQVRFU19QQVRIfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "src/tokenization/train_tokenizer.py": "aW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHRva2VuaXplcnMgaW1wb3J0IFRva2VuaXplciwgZGVjb2RlcnMsIG1vZGVscywgcHJlX3Rva2VuaXplcnMsIHByb2Nlc3NvcnMsIHRyYWluZXJzCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgKAogICAgRU5fVk9DQUJfU0laRSwKICAgIEVPU19JRCwKICAgIE9SX1ZPQ0FCX1NJWkUsCiAgICBQQURfSUQsCiAgICBTT1NfSUQsCiAgICBTUEVDSUFMX1RPS0VOUywKICAgIFRPS0VOSVpFUl9ESVIsCiAgICBVTktfSUQsCikKZnJvbSBzcmMuZGF0YS5kb3dubG9hZCBpbXBvcnQgQ0FORElEQVRFU19QQVRICgpFTl9UT0tFTklaRVJfUEFUSCA9IFRPS0VOSVpFUl9ESVIgLyAiZW5fYnBlLmpzb24iCk9SX1RPS0VOSVpFUl9QQVRIID0gVE9LRU5JWkVSX0RJUiAvICJvcl9icGUuanNvbiIKCgpkZWYgX3RyYWluX29uZSh0ZXh0czogbGlzdFtzdHJdLCB2b2NhYl9zaXplOiBpbnQpIC0+IFRva2VuaXplcjoKICAgIHRvayA9IFRva2VuaXplcihtb2RlbHMuQlBFKHVua190b2tlbj0iPFVOSz4iKSkKICAgIHRvay5wcmVfdG9rZW5pemVyID0gcHJlX3Rva2VuaXplcnMuQnl0ZUxldmVsKGFkZF9wcmVmaXhfc3BhY2U9VHJ1ZSkKICAgIHRvay5kZWNvZGVyID0gZGVjb2RlcnMuQnl0ZUxldmVsKCkKCiAgICB0cmFpbmVyID0gdHJhaW5lcnMuQnBlVHJhaW5lcih2b2NhYl9zaXplPXZvY2FiX3NpemUsIHNwZWNpYWxfdG9rZW5zPVNQRUNJQUxfVE9LRU5TKQogICAgdG9rLnRyYWluX2Zyb21faXRlcmF0b3IodGV4dHMsIHRyYWluZXI9dHJhaW5lcikKCiAgICB0b2sucG9zdF9wcm9jZXNzb3IgPSBwcm9jZXNzb3JzLlRlbXBsYXRlUHJvY2Vzc2luZygKICAgICAgICBzaW5nbGU9IjxTT1M+ICRBIDxFT1M+IiwKICAgICAgICBzcGVjaWFsX3Rva2Vucz1bKCI8U09TPiIsIHRvay50b2tlbl90b19pZCgiPFNPUz4iKSksICgiPEVPUz4iLCB0b2sudG9rZW5fdG9faWQoIjxFT1M+IikpXSwKICAgICkKCiAgICBhc3NlcnQgdG9rLnRva2VuX3RvX2lkKCI8UEFEPiIpID09IFBBRF9JRAogICAgYXNzZXJ0IHRvay50b2tlbl90b19pZCgiPFNPUz4iKSA9PSBTT1NfSUQKICAgIGFzc2VydCB0b2sudG9rZW5fdG9faWQoIjxFT1M+IikgPT0gRU9TX0lECiAgICBhc3NlcnQgdG9rLnRva2VuX3RvX2lkKCI8VU5LPiIpID09IFVOS19JRAoKICAgIHJldHVybiB0b2sKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBUT0tFTklaRVJfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpZiBFTl9UT0tFTklaRVJfUEFUSC5leGlzdHMoKSBhbmQgT1JfVE9LRU5JWkVSX1BBVEguZXhpc3RzKCk6CiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYie0VOX1RPS0VOSVpFUl9QQVRIfSBhbmQge09SX1RPS0VOSVpFUl9QQVRIfSBhbHJlYWR5IGV4aXN0OyBza2lwcGluZyB0cmFpbmluZy4gIgogICAgICAgICAgICAiRGVsZXRlIHRoZW0gdG8gZm9yY2UgYSByZXRyYWluLiIKICAgICAgICApCiAgICAgICAgcmV0dXJuCgogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQoQ0FORElEQVRFU19QQVRIKQoKICAgIGVuX3RvayA9IF90cmFpbl9vbmUoZGZbInNyYyJdLnRvbGlzdCgpLCBFTl9WT0NBQl9TSVpFKQogICAgZW5fdG9rLnNhdmUoc3RyKEVOX1RPS0VOSVpFUl9QQVRIKSkKICAgIHByaW50KGYic2F2ZWQgRW5nbGlzaCB0b2tlbml6ZXIgKHtlbl90b2suZ2V0X3ZvY2FiX3NpemUoKX0gdG9rZW5zKSB0byB7RU5fVE9LRU5JWkVSX1BBVEh9IikKCiAgICBvcl90b2sgPSBfdHJhaW5fb25lKGRmWyJ0Z3QiXS50b2xpc3QoKSwgT1JfVk9DQUJfU0laRSkKICAgIG9yX3Rvay5zYXZlKHN0cihPUl9UT0tFTklaRVJfUEFUSCkpCiAgICBwcmludChmInNhdmVkIE9kaWEgdG9rZW5pemVyICh7b3JfdG9rLmdldF92b2NhYl9zaXplKCl9IHRva2VucykgdG8ge09SX1RPS0VOSVpFUl9QQVRIfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "src/tokenization/tokenizer_utils.py": "ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHRva2VuaXplcnMgaW1wb3J0IFRva2VuaXplcgoKCmRlZiBsb2FkX3Rva2VuaXplcihwYXRoOiBQYXRoKSAtPiBUb2tlbml6ZXI6CiAgICByZXR1cm4gVG9rZW5pemVyLmZyb21fZmlsZShzdHIocGF0aCkpCgoKZGVmIGVuY29kZSh0b2s6IFRva2VuaXplciwgdGV4dDogc3RyKSAtPiBsaXN0W2ludF06CiAgICByZXR1cm4gdG9rLmVuY29kZSh0ZXh0KS5pZHMKCgpkZWYgZGVjb2RlKHRvazogVG9rZW5pemVyLCBpZHM6IGxpc3RbaW50XSkgLT4gc3RyOgogICAgdGV4dCA9IHRvay5kZWNvZGUoaWRzLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICAjIGFkZF9wcmVmaXhfc3BhY2U9VHJ1ZSBvbiB0aGUgcHJlLXRva2VuaXplciBpbmplY3RzIGEgc3ludGhldGljIGxlYWRpbmcKICAgICMgc3BhY2UgYmVmb3JlIHRva2VuaXphdGlvbiBzbyB0aGUgZmlyc3Qgd29yZCBpcyB0cmVhdGVkIGxpa2UgYW55IG90aGVyOwogICAgIyB0aGF0IHNwYWNlIHJvdW5kLXRyaXBzIGJhY2sgb24gZGVjb2RlIGFuZCBtdXN0IGJlIGRyb3BwZWQgaGVyZQogICAgaWYgdGV4dC5zdGFydHN3aXRoKCIgIik6CiAgICAgICAgdGV4dCA9IHRleHRbMTpdCiAgICByZXR1cm4gdGV4dAo=", "src/data/split.py": "aW1wb3J0IHJhbmRvbQppbXBvcnQgc3lzCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgKAogICAgREFUQV9QUk9DRVNTRURfRElSLAogICAgTUFYX0xFTiwKICAgIFJBTkRPTV9TRUVELAogICAgVEVTVF9TSVpFLAogICAgVE9UQUxfU0laRSwKICAgIFRSQUlOX1NJWkUsCiAgICBWQUxfU0laRSwKKQpmcm9tIHNyYy5kYXRhLmRvd25sb2FkIGltcG9ydCBDQU5ESURBVEVTX1BBVEgKZnJvbSBzcmMudG9rZW5pemF0aW9uLnRyYWluX3Rva2VuaXplciBpbXBvcnQgRU5fVE9LRU5JWkVSX1BBVEgsIE9SX1RPS0VOSVpFUl9QQVRICmZyb20gc3JjLnRva2VuaXphdGlvbi50b2tlbml6ZXJfdXRpbHMgaW1wb3J0IGVuY29kZSwgbG9hZF90b2tlbml6ZXIKCkVTVElNQVRFRF9SRVRFTlRJT04gPSAwLjc4MAoKVFJBSU5fUEFUSCA9IERBVEFfUFJPQ0VTU0VEX0RJUiAvICJ0cmFpbi5wYXJxdWV0IgpWQUxfUEFUSCA9IERBVEFfUFJPQ0VTU0VEX0RJUiAvICJ2YWwucGFycXVldCIKVEVTVF9QQVRIID0gREFUQV9QUk9DRVNTRURfRElSIC8gInRlc3QucGFycXVldCIKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBEQVRBX1BST0NFU1NFRF9ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGNhbmRpZGF0ZXMgPSBwZC5yZWFkX3BhcnF1ZXQoQ0FORElEQVRFU19QQVRIKQogICAgcG9vbF9zaXplID0gbGVuKGNhbmRpZGF0ZXMpCgogICAgZW5fdG9rID0gbG9hZF90b2tlbml6ZXIoRU5fVE9LRU5JWkVSX1BBVEgpCiAgICBvcl90b2sgPSBsb2FkX3Rva2VuaXplcihPUl9UT0tFTklaRVJfUEFUSCkKCiAgICBrZXB0X3Jvd3MgPSBbXQogICAgZm9yIHNyYywgdGd0IGluIHppcChjYW5kaWRhdGVzWyJzcmMiXSwgY2FuZGlkYXRlc1sidGd0Il0pOgogICAgICAgIGVuX2lkcyA9IGVuY29kZShlbl90b2ssIHNyYykKICAgICAgICBvcl9pZHMgPSBlbmNvZGUob3JfdG9rLCB0Z3QpCiAgICAgICAgaWYgbGVuKGVuX2lkcykgPD0gTUFYX0xFTiBhbmQgbGVuKG9yX2lkcykgPD0gTUFYX0xFTjoKICAgICAgICAgICAga2VwdF9yb3dzLmFwcGVuZCh7InNyYyI6IHNyYywgInRndCI6IHRndH0pCgogICAgc3Vydml2b3JzID0gcGQuRGF0YUZyYW1lKGtlcHRfcm93cywgY29sdW1ucz1bInNyYyIsICJ0Z3QiXSkKICAgIHN1cnZpdm9ycyA9IHN1cnZpdm9ycy5kcm9wX2R1cGxpY2F0ZXMoc3Vic2V0PSJzcmMiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBzdXJ2aXZvcl9jb3VudCA9IGxlbihzdXJ2aXZvcnMpCiAgICByZXRlbnRpb25fcmF0ZSA9IHN1cnZpdm9yX2NvdW50IC8gcG9vbF9zaXplIGlmIHBvb2xfc2l6ZSBlbHNlIDAuMAoKICAgIGlmIHN1cnZpdm9yX2NvdW50IDwgVE9UQUxfU0laRToKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJXQVJOSU5HOiBvbmx5IHtzdXJ2aXZvcl9jb3VudH0gcGFpcnMgc3Vydml2ZWQgTUFYX0xFTj17TUFYX0xFTn0gZmlsdGVyaW5nLCAiCiAgICAgICAgICAgIGYiYmVsb3cgVE9UQUxfU0laRT17VE9UQUxfU0laRX0uIE9ic2VydmVkIHJldGVudGlvbiByYXRlIHtyZXRlbnRpb25fcmF0ZTouMSV9IHZzICIKICAgICAgICAgICAgZiJ0aGUge0VTVElNQVRFRF9SRVRFTlRJT046LjElfSBlc3RpbWF0ZS4gSW5jcmVhc2UgQ0FORElEQVRFX1BPT0xfU0laRSBhbmQgcmUtcnVuICIKICAgICAgICAgICAgImRvd25sb2FkL3RyYWluX3Rva2VuaXplci9zcGxpdC4iLAogICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgKQogICAgICAgIGZpbmFsID0gc3Vydml2b3JzCiAgICBlbHNlOgogICAgICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oUkFORE9NX1NFRUQpCiAgICAgICAgaW5kaWNlcyA9IGxpc3QocmFuZ2Uoc3Vydml2b3JfY291bnQpKQogICAgICAgIHJuZy5zaHVmZmxlKGluZGljZXMpCiAgICAgICAgZmluYWwgPSBzdXJ2aXZvcnMuaWxvY1tpbmRpY2VzWzpUT1RBTF9TSVpFXV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKICAgIGZpbmFsX2NvdW50ID0gbGVuKGZpbmFsKQogICAgbl90cmFpbiA9IG1pbihUUkFJTl9TSVpFLCBmaW5hbF9jb3VudCkKICAgIG5fdmFsID0gbWluKFZBTF9TSVpFLCBtYXgoZmluYWxfY291bnQgLSBuX3RyYWluLCAwKSkKICAgIG5fdGVzdCA9IG1heChmaW5hbF9jb3VudCAtIG5fdHJhaW4gLSBuX3ZhbCwgMCkKCiAgICB0cmFpbl9kZiA9IGZpbmFsLmlsb2NbOm5fdHJhaW5dLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHZhbF9kZiA9IGZpbmFsLmlsb2Nbbl90cmFpbjpuX3RyYWluICsgbl92YWxdLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHRlc3RfZGYgPSBmaW5hbC5pbG9jW25fdHJhaW4gKyBuX3ZhbDpuX3RyYWluICsgbl92YWwgKyBuX3Rlc3RdLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCiAgICB0cmFpbl9zcmMgPSBzZXQodHJhaW5fZGZbInNyYyJdKQogICAgdmFsX3NyYyA9IHNldCh2YWxfZGZbInNyYyJdKQogICAgdGVzdF9zcmMgPSBzZXQodGVzdF9kZlsic3JjIl0pCiAgICBhc3NlcnQgbm90ICh0cmFpbl9zcmMgJiB2YWxfc3JjKQogICAgYXNzZXJ0IG5vdCAodHJhaW5fc3JjICYgdGVzdF9zcmMpCiAgICBhc3NlcnQgbm90ICh2YWxfc3JjICYgdGVzdF9zcmMpCgogICAgdHJhaW5fZGYudG9fcGFycXVldChUUkFJTl9QQVRILCBpbmRleD1GYWxzZSkKICAgIHZhbF9kZi50b19wYXJxdWV0KFZBTF9QQVRILCBpbmRleD1GYWxzZSkKICAgIHRlc3RfZGYudG9fcGFycXVldChURVNUX1BBVEgsIGluZGV4PUZhbHNlKQoKICAgIHByaW50KGYiY2FuZGlkYXRlIHBvb2wgc2l6ZToge3Bvb2xfc2l6ZX0iKQogICAgcHJpbnQoZiJwb3N0LWZpbHRlciBzdXJ2aXZvciBjb3VudCAoTUFYX0xFTj17TUFYX0xFTn0pOiB7c3Vydml2b3JfY291bnR9IikKICAgIHByaW50KAogICAgICAgIGYib2JzZXJ2ZWQgcmV0ZW50aW9uIHJhdGU6IHtyZXRlbnRpb25fcmF0ZTouMSV9ICIKICAgICAgICBmIih0aHJvd2F3YXktc2FtcGxlIGVzdGltYXRlIHdhcyB7RVNUSU1BVEVEX1JFVEVOVElPTjouMSV9KSIKICAgICkKICAgIHByaW50KGYidHJhaW46IHtsZW4odHJhaW5fZGYpfSwgdmFsOiB7bGVuKHZhbF9kZil9LCB0ZXN0OiB7bGVuKHRlc3RfZGYpfSIpCiAgICBwcmludChmInNhdmVkIHRvIHtUUkFJTl9QQVRIfSwge1ZBTF9QQVRIfSwge1RFU1RfUEFUSH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "src/data/dataset.py": "ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCmZyb20gY29uZmlncy5iYXNlIGltcG9ydCBQQURfSUQKZnJvbSBzcmMudG9rZW5pemF0aW9uLnRyYWluX3Rva2VuaXplciBpbXBvcnQgRU5fVE9LRU5JWkVSX1BBVEgsIE9SX1RPS0VOSVpFUl9QQVRICmZyb20gc3JjLnRva2VuaXphdGlvbi50b2tlbml6ZXJfdXRpbHMgaW1wb3J0IGVuY29kZSwgbG9hZF90b2tlbml6ZXIKCgpjbGFzcyBUcmFuc2xhdGlvbkRhdGFzZXQoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBzcGxpdF9wYXRoOiBQYXRoLAogICAgICAgIGVuX3Rva2VuaXplcl9wYXRoOiBQYXRoID0gRU5fVE9LRU5JWkVSX1BBVEgsCiAgICAgICAgb3JfdG9rZW5pemVyX3BhdGg6IFBhdGggPSBPUl9UT0tFTklaRVJfUEFUSCwKICAgICk6CiAgICAgICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQoc3BsaXRfcGF0aCkKICAgICAgICBlbl90b2sgPSBsb2FkX3Rva2VuaXplcihlbl90b2tlbml6ZXJfcGF0aCkKICAgICAgICBvcl90b2sgPSBsb2FkX3Rva2VuaXplcihvcl90b2tlbml6ZXJfcGF0aCkKCiAgICAgICAgc2VsZi5zcmNfaWRzID0gW3RvcmNoLnRlbnNvcihlbmNvZGUoZW5fdG9rLCBzKSwgZHR5cGU9dG9yY2gubG9uZykgZm9yIHMgaW4gZGZbInNyYyJdXQogICAgICAgIHNlbGYudGd0X2lkcyA9IFt0b3JjaC50ZW5zb3IoZW5jb2RlKG9yX3RvaywgdCksIGR0eXBlPXRvcmNoLmxvbmcpIGZvciB0IGluIGRmWyJ0Z3QiXV0KCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLnNyY19pZHMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KToKICAgICAgICByZXR1cm4gc2VsZi5zcmNfaWRzW2lkeF0sIHNlbGYudGd0X2lkc1tpZHhdCgoKZGVmIF9wYWRfYmF0Y2goc2VxdWVuY2VzOiBsaXN0W3RvcmNoLlRlbnNvcl0pIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yXToKICAgIG1heF9sZW4gPSBtYXgoc2VxLnNpemUoMCkgZm9yIHNlcSBpbiBzZXF1ZW5jZXMpCiAgICBwYWRkZWQgPSB0b3JjaC5mdWxsKChsZW4oc2VxdWVuY2VzKSwgbWF4X2xlbiksIFBBRF9JRCwgZHR5cGU9dG9yY2gubG9uZykKICAgICMgcGFkX21hc2s6IFRydWUgYXQgUEFEIHBvc2l0aW9ucywgbWF0Y2hpbmcgbm4uTXVsdGloZWFkQXR0ZW50aW9uJ3MKICAgICMga2V5X3BhZGRpbmdfbWFzayBjb252ZW50aW9uIHdoZXJlIFRydWUgcG9zaXRpb25zIGFyZSBpZ25vcmVkCiAgICBwYWRfbWFzayA9IHRvcmNoLm9uZXMoKGxlbihzZXF1ZW5jZXMpLCBtYXhfbGVuKSwgZHR5cGU9dG9yY2guYm9vbCkKICAgIGZvciBpLCBzZXEgaW4gZW51bWVyYXRlKHNlcXVlbmNlcyk6CiAgICAgICAgcGFkZGVkW2ksIDpzZXEuc2l6ZSgwKV0gPSBzZXEKICAgICAgICBwYWRfbWFza1tpLCA6c2VxLnNpemUoMCldID0gRmFsc2UKICAgIHJldHVybiBwYWRkZWQsIHBhZF9tYXNrCgoKZGVmIGNvbGxhdGVfZm4oYmF0Y2g6IGxpc3RbdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdXSk6CiAgICBzcmNfc2VxcywgdGd0X3NlcXMgPSB6aXAoKmJhdGNoKQogICAgc3JjX2lkcywgc3JjX3BhZF9tYXNrID0gX3BhZF9iYXRjaChsaXN0KHNyY19zZXFzKSkKICAgIHRndF9pZHMsIHRndF9wYWRfbWFzayA9IF9wYWRfYmF0Y2gobGlzdCh0Z3Rfc2VxcykpCiAgICByZXR1cm4gc3JjX2lkcywgc3JjX3BhZF9tYXNrLCB0Z3RfaWRzLCB0Z3RfcGFkX21hc2sK", "src/model/masks.py": "IiIiQXR0ZW50aW9uIG1hc2tzIGZvciB0aGUgZW5jb2Rlci1kZWNvZGVyIHRyYW5zZm9ybWVyLgoKQ29udmVudGlvbiB1c2VkIHRocm91Z2hvdXQgdGhpcyBtb2R1bGUgYW5kIGNvbnN1bWVkIGJ5IGF0dGVudGlvbi5weToKbWFza3MgYXJlIGJvb2xlYW4gdGVuc29ycyB3aGVyZSBUcnVlIG1lYW5zICJhdHRlbmQgLyBrZWVwIiBhbmQgRmFsc2UKbWVhbnMgIm1hc2tlZCBvdXQiLiBUaGUgYXR0ZW50aW9uIG1vZHVsZSBpcyByZXNwb25zaWJsZSBmb3IgdHVybmluZwpGYWxzZSBwb3NpdGlvbnMgaW50byBhIGxhcmdlIG5lZ2F0aXZlIGJpYXMgYmVmb3JlIHNvZnRtYXguCiIiIgoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKCmRlZiBtYWtlX3BhZGRpbmdfbWFzayhpZHM6IFRlbnNvciwgcGFkX2lkOiBpbnQpIC0+IFRlbnNvcjoKICAgIHJldHVybiAoaWRzICE9IHBhZF9pZCkudW5zcXVlZXplKDEpLnVuc3F1ZWV6ZSgxKQoKCmRlZiBtYWtlX2NhdXNhbF9tYXNrKHNpemU6IGludCkgLT4gVGVuc29yOgogICAgcmV0dXJuIHRvcmNoLnRyaWwodG9yY2gub25lcyhzaXplLCBzaXplLCBkdHlwZT10b3JjaC5ib29sKSkKCgpkZWYgbWFrZV9kZWNvZGVyX3NlbGZfYXR0bl9tYXNrKHRndF9pZHM6IFRlbnNvciwgcGFkX2lkOiBpbnQpIC0+IFRlbnNvcjoKICAgIFQgPSB0Z3RfaWRzLnNpemUoMSkKICAgIGNhdXNhbCA9IG1ha2VfY2F1c2FsX21hc2soVCkudG8odGd0X2lkcy5kZXZpY2UpCiAgICBwYWRkaW5nID0gbWFrZV9wYWRkaW5nX21hc2sodGd0X2lkcywgcGFkX2lkKQogICAgcmV0dXJuIGNhdXNhbC51bnNxdWVlemUoMCkudW5zcXVlZXplKDApICYgcGFkZGluZwoKCmRlZiBtYWtlX2Nyb3NzX2F0dG5fbWFzayhzcmNfaWRzOiBUZW5zb3IsIHBhZF9pZDogaW50KSAtPiBUZW5zb3I6CiAgICByZXR1cm4gbWFrZV9wYWRkaW5nX21hc2soc3JjX2lkcywgcGFkX2lkKQoKCiMgTmFOLWZyb20tYWxsLW1hc2tlZC1yb3cgbm90ZTogc29mdG1heCBvdmVyIGEgcm93IHRoYXQgaXMgZW50aXJlbHkKIyBtYXNrZWQgKGFsbCBGYWxzZSkgd291bGQgcHJvZHVjZSBOYU4sIHNpbmNlIGV2ZXJ5IGxvZ2l0IGJlY29tZXMgLWluZgojIGFuZCBleHAoLWluZikgc3VtcyB0byB6ZXJvLiBUaGF0IGNhbid0IGhhcHBlbiBmb3IgdGhlIG1hc2sgc2hhcGVzCiMgYnVpbHQgaGVyZSBhcyBsb25nIGFzIGV2ZXJ5IGV4YW1wbGUgaGFzIGF0IGxlYXN0IG9uZSBub24tcGFkIHRva2VuLgojIERlY29kZXIgc2VsZi1hdHRlbnRpb246IHF1ZXJ5IHBvc2l0aW9uIGkgaXMgY2F1c2FsbHkgYWxsb3dlZCB0byBzZWUKIyBrZXkgcG9zaXRpb24gaSwgYW5kIHJlYWwgKG5vbi1wYWQpIHRhcmdldCBzZXF1ZW5jZXMgYWx3YXlzIGhhdmUgYQojIG5vbi1wYWQgdG9rZW4gYXQgcG9zaXRpb24gMCAoPFNPUz4pLCBzbyByb3cgaSBmb3IgYW55IHJlYWwgcXVlcnkKIyBhbHdheXMgaGFzIGF0IGxlYXN0IG9uZSB1bm1hc2tlZCBrZXkgKGl0c2VsZiBvciBhbiBlYXJsaWVyIHJlYWwKIyB0b2tlbikuIENyb3NzLWF0dGVudGlvbjogZXZlcnkgcm93IGlzIG1hc2tlZCBpZGVudGljYWxseSBieSB0aGUKIyBzb3VyY2UgcGFkZGluZyBwYXR0ZXJuLCBzbyBhcyBsb25nIGFzIHRoZSBzb3VyY2Ugc2VxdWVuY2UgaGFzIGF0CiMgbGVhc3Qgb25lIG5vbi1wYWQgdG9rZW4sIG5vIHJvdyBpcyBmdWxseSBtYXNrZWQuIEZ1bGx5LWVtcHR5CiMgc2VxdWVuY2VzIChhbGwgcGFkKSBhcmUgbm90IGEgY2FzZSB0aGlzIGNvZGViYXNlIHByb2R1Y2VzLgo=", "src/model/embeddings.py": "aW1wb3J0IG1hdGgKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgRF9NT0RFTCwgRFJPUE9VVCwgTUFYX0xFTgoKCmNsYXNzIFBvc2l0aW9uYWxFbmNvZGluZyhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWw6IGludCA9IERfTU9ERUwsIGRyb3BvdXQ6IGZsb2F0ID0gRFJPUE9VVCwgbWF4X2xlbjogaW50ID0gTUFYX0xFTiArIDE2KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGRyb3BvdXQpCgogICAgICAgIHBlID0gdG9yY2guemVyb3MobWF4X2xlbiwgZF9tb2RlbCkKICAgICAgICBwb3NpdGlvbiA9IHRvcmNoLmFyYW5nZSgwLCBtYXhfbGVuLCBkdHlwZT10b3JjaC5mbG9hdDMyKS51bnNxdWVlemUoMSkKICAgICAgICBkaXZfdGVybSA9IHRvcmNoLmV4cCgKICAgICAgICAgICAgdG9yY2guYXJhbmdlKDAsIGRfbW9kZWwsIDIsIGR0eXBlPXRvcmNoLmZsb2F0MzIpICogKC1tYXRoLmxvZygxMDAwMC4wKSAvIGRfbW9kZWwpCiAgICAgICAgKQogICAgICAgIHBlWzosIDA6OjJdID0gdG9yY2guc2luKHBvc2l0aW9uICogZGl2X3Rlcm0pCiAgICAgICAgcGVbOiwgMTo6Ml0gPSB0b3JjaC5jb3MocG9zaXRpb24gKiBkaXZfdGVybSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigicGUiLCBwZSwgcGVyc2lzdGVudD1GYWxzZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgICAgICBzZXFfbGVuID0geC5zaXplKDEpCiAgICAgICAgeCA9IHggKyBzZWxmLnBlWzpzZXFfbGVuLCA6XS51bnNxdWVlemUoMCkKICAgICAgICByZXR1cm4gc2VsZi5kcm9wb3V0KHgpCgoKY2xhc3MgVG9rZW5FbWJlZGRpbmcobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2b2NhYl9zaXplOiBpbnQsIGRfbW9kZWw6IGludCA9IERfTU9ERUwpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZW1iZWRkaW5nID0gbm4uRW1iZWRkaW5nKHZvY2FiX3NpemUsIGRfbW9kZWwpCiAgICAgICAgc2VsZi5kX21vZGVsID0gZF9tb2RlbAoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlkczogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYuZW1iZWRkaW5nKGlkcykgKiBtYXRoLnNxcnQoc2VsZi5kX21vZGVsKQoKCmNsYXNzIEVtYmVkZGluZ3Mobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHZvY2FiX3NpemU6IGludCwKICAgICAgICBkX21vZGVsOiBpbnQgPSBEX01PREVMLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gRFJPUE9VVCwKICAgICAgICBtYXhfbGVuOiBpbnQgPSBNQVhfTEVOICsgMTYsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYudG9rZW5fZW1iZWRkaW5nID0gVG9rZW5FbWJlZGRpbmcodm9jYWJfc2l6ZSwgZF9tb2RlbCkKICAgICAgICBzZWxmLnBvc2l0aW9uYWxfZW5jb2RpbmcgPSBQb3NpdGlvbmFsRW5jb2RpbmcoZF9tb2RlbCwgZHJvcG91dCwgbWF4X2xlbikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpZHM6IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIHJldHVybiBzZWxmLnBvc2l0aW9uYWxfZW5jb2Rpbmcoc2VsZi50b2tlbl9lbWJlZGRpbmcoaWRzKSkK", "src/model/attention.py": "aW1wb3J0IG1hdGgKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgRF9NT0RFTCwgTl9IRUFEUwoKX01BU0tfRklMTF9WQUxVRSA9IC0xZTkKCgpjbGFzcyBNdWx0aUhlYWRBdHRlbnRpb24obm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkX21vZGVsOiBpbnQgPSBEX01PREVMLCBuX2hlYWRzOiBpbnQgPSBOX0hFQURTKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBhc3NlcnQgZF9tb2RlbCAlIG5faGVhZHMgPT0gMAoKICAgICAgICBzZWxmLmRfbW9kZWwgPSBkX21vZGVsCiAgICAgICAgc2VsZi5uX2hlYWRzID0gbl9oZWFkcwogICAgICAgIHNlbGYuaGVhZF9kaW0gPSBkX21vZGVsIC8vIG5faGVhZHMKCiAgICAgICAgc2VsZi5xX3Byb2ogPSBubi5MaW5lYXIoZF9tb2RlbCwgZF9tb2RlbCkKICAgICAgICBzZWxmLmtfcHJvaiA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsKQogICAgICAgIHNlbGYudl9wcm9qID0gbm4uTGluZWFyKGRfbW9kZWwsIGRfbW9kZWwpCiAgICAgICAgc2VsZi5vdXRfcHJvaiA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsKQoKICAgIGRlZiBfc3BsaXRfaGVhZHMoc2VsZiwgeDogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgQiwgTCwgXyA9IHguc2hhcGUKICAgICAgICB4ID0geC52aWV3KEIsIEwsIHNlbGYubl9oZWFkcywgc2VsZi5oZWFkX2RpbSkKICAgICAgICByZXR1cm4geC5wZXJtdXRlKDAsIDIsIDEsIDMpCgogICAgZGVmIF9tZXJnZV9oZWFkcyhzZWxmLCB4OiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgICAgICBCLCBILCBMLCBEID0geC5zaGFwZQogICAgICAgIHggPSB4LnBlcm11dGUoMCwgMiwgMSwgMykuY29udGlndW91cygpCiAgICAgICAgcmV0dXJuIHgudmlldyhCLCBMLCBIICogRCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBxdWVyeTogVGVuc29yLCBrZXk6IFRlbnNvciwgdmFsdWU6IFRlbnNvciwgbWFzazogVGVuc29yID0gTm9uZSkgLT4gVGVuc29yOgogICAgICAgIHEgPSBzZWxmLl9zcGxpdF9oZWFkcyhzZWxmLnFfcHJvaihxdWVyeSkpCiAgICAgICAgayA9IHNlbGYuX3NwbGl0X2hlYWRzKHNlbGYua19wcm9qKGtleSkpCiAgICAgICAgdiA9IHNlbGYuX3NwbGl0X2hlYWRzKHNlbGYudl9wcm9qKHZhbHVlKSkKCiAgICAgICAgc2NvcmVzID0gdG9yY2gubWF0bXVsKHEsIGsudHJhbnNwb3NlKC0yLCAtMSkpIC8gbWF0aC5zcXJ0KHNlbGYuaGVhZF9kaW0pCgogICAgICAgIGlmIG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNjb3JlcyA9IHNjb3Jlcy5tYXNrZWRfZmlsbCh+bWFzaywgX01BU0tfRklMTF9WQUxVRSkKCiAgICAgICAgYXR0biA9IHRvcmNoLnNvZnRtYXgoc2NvcmVzLCBkaW09LTEpCiAgICAgICAgb3V0ID0gdG9yY2gubWF0bXVsKGF0dG4sIHYpCiAgICAgICAgb3V0ID0gc2VsZi5fbWVyZ2VfaGVhZHMob3V0KQogICAgICAgIHJldHVybiBzZWxmLm91dF9wcm9qKG91dCkK", "src/model/feedforward.py": "aW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0IERfRkYsIERfTU9ERUwsIERST1BPVVQKCgpjbGFzcyBQb3NpdGlvbndpc2VGZWVkRm9yd2FyZChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWw6IGludCA9IERfTU9ERUwsIGRfZmY6IGludCA9IERfRkYsIGRyb3BvdXQ6IGZsb2F0ID0gRFJPUE9VVCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5saW5lYXIxID0gbm4uTGluZWFyKGRfbW9kZWwsIGRfZmYpCiAgICAgICAgc2VsZi5yZWx1ID0gbm4uUmVMVSgpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYubGluZWFyMiA9IG5uLkxpbmVhcihkX2ZmLCBkX21vZGVsKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIHggPSBzZWxmLmxpbmVhcjEoeCkKICAgICAgICB4ID0gc2VsZi5yZWx1KHgpCiAgICAgICAgeCA9IHNlbGYuZHJvcG91dCh4KQogICAgICAgIHJldHVybiBzZWxmLmxpbmVhcjIoeCkK", "src/model/encoder.py": "aW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0IERfRkYsIERfTU9ERUwsIERST1BPVVQsIE5fRU5DT0RFUl9MQVlFUlMsIE5fSEVBRFMKZnJvbSBzcmMubW9kZWwuYXR0ZW50aW9uIGltcG9ydCBNdWx0aUhlYWRBdHRlbnRpb24KZnJvbSBzcmMubW9kZWwuZW1iZWRkaW5ncyBpbXBvcnQgRW1iZWRkaW5ncwpmcm9tIHNyYy5tb2RlbC5mZWVkZm9yd2FyZCBpbXBvcnQgUG9zaXRpb253aXNlRmVlZEZvcndhcmQKCgpjbGFzcyBFbmNvZGVyQmxvY2sobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGRfbW9kZWw6IGludCA9IERfTU9ERUwsCiAgICAgICAgbl9oZWFkczogaW50ID0gTl9IRUFEUywKICAgICAgICBkX2ZmOiBpbnQgPSBEX0ZGLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gRFJPUE9VVCwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5zZWxmX2F0dG4gPSBNdWx0aUhlYWRBdHRlbnRpb24oZF9tb2RlbCwgbl9oZWFkcykKICAgICAgICBzZWxmLmRyb3BvdXQxID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYubm9ybTEgPSBubi5MYXllck5vcm0oZF9tb2RlbCkKCiAgICAgICAgc2VsZi5mZWVkX2ZvcndhcmQgPSBQb3NpdGlvbndpc2VGZWVkRm9yd2FyZChkX21vZGVsLCBkX2ZmLCBkcm9wb3V0KQogICAgICAgIHNlbGYuZHJvcG91dDIgPSBubi5Ecm9wb3V0KGRyb3BvdXQpCiAgICAgICAgc2VsZi5ub3JtMiA9IG5uLkxheWVyTm9ybShkX21vZGVsKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IFRlbnNvciwgc3JjX21hc2s6IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIGF0dG5fb3V0ID0gc2VsZi5zZWxmX2F0dG4oeCwgeCwgeCwgc3JjX21hc2spCiAgICAgICAgeCA9IHNlbGYubm9ybTEoeCArIHNlbGYuZHJvcG91dDEoYXR0bl9vdXQpKQoKICAgICAgICBmZl9vdXQgPSBzZWxmLmZlZWRfZm9yd2FyZCh4KQogICAgICAgIHggPSBzZWxmLm5vcm0yKHggKyBzZWxmLmRyb3BvdXQyKGZmX291dCkpCiAgICAgICAgcmV0dXJuIHgKCgpjbGFzcyBFbmNvZGVyKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICB2b2NhYl9zaXplOiBpbnQsCiAgICAgICAgZF9tb2RlbDogaW50ID0gRF9NT0RFTCwKICAgICAgICBuX2hlYWRzOiBpbnQgPSBOX0hFQURTLAogICAgICAgIGRfZmY6IGludCA9IERfRkYsCiAgICAgICAgbl9sYXllcnM6IGludCA9IE5fRU5DT0RFUl9MQVlFUlMsCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSBEUk9QT1VULAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmVtYmVkZGluZ3MgPSBFbWJlZGRpbmdzKHZvY2FiX3NpemUsIGRfbW9kZWwsIGRyb3BvdXQpCiAgICAgICAgc2VsZi5sYXllcnMgPSBubi5Nb2R1bGVMaXN0KAogICAgICAgICAgICBbRW5jb2RlckJsb2NrKGRfbW9kZWwsIG5faGVhZHMsIGRfZmYsIGRyb3BvdXQpIGZvciBfIGluIHJhbmdlKG5fbGF5ZXJzKV0KICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgc3JjX2lkczogVGVuc29yLCBzcmNfbWFzazogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgeCA9IHNlbGYuZW1iZWRkaW5ncyhzcmNfaWRzKQogICAgICAgIGZvciBsYXllciBpbiBzZWxmLmxheWVyczoKICAgICAgICAgICAgeCA9IGxheWVyKHgsIHNyY19tYXNrKQogICAgICAgIHJldHVybiB4Cg==", "src/model/decoder.py": "aW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0IERfRkYsIERfTU9ERUwsIERST1BPVVQsIE5fREVDT0RFUl9MQVlFUlMsIE5fSEVBRFMKZnJvbSBzcmMubW9kZWwuYXR0ZW50aW9uIGltcG9ydCBNdWx0aUhlYWRBdHRlbnRpb24KZnJvbSBzcmMubW9kZWwuZW1iZWRkaW5ncyBpbXBvcnQgRW1iZWRkaW5ncwpmcm9tIHNyYy5tb2RlbC5mZWVkZm9yd2FyZCBpbXBvcnQgUG9zaXRpb253aXNlRmVlZEZvcndhcmQKCgpjbGFzcyBEZWNvZGVyQmxvY2sobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGRfbW9kZWw6IGludCA9IERfTU9ERUwsCiAgICAgICAgbl9oZWFkczogaW50ID0gTl9IRUFEUywKICAgICAgICBkX2ZmOiBpbnQgPSBEX0ZGLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gRFJPUE9VVCwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5zZWxmX2F0dG4gPSBNdWx0aUhlYWRBdHRlbnRpb24oZF9tb2RlbCwgbl9oZWFkcykKICAgICAgICBzZWxmLmRyb3BvdXQxID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYubm9ybTEgPSBubi5MYXllck5vcm0oZF9tb2RlbCkKCiAgICAgICAgc2VsZi5jcm9zc19hdHRuID0gTXVsdGlIZWFkQXR0ZW50aW9uKGRfbW9kZWwsIG5faGVhZHMpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZHJvcG91dCkKICAgICAgICBzZWxmLm5vcm0yID0gbm4uTGF5ZXJOb3JtKGRfbW9kZWwpCgogICAgICAgIHNlbGYuZmVlZF9mb3J3YXJkID0gUG9zaXRpb253aXNlRmVlZEZvcndhcmQoZF9tb2RlbCwgZF9mZiwgZHJvcG91dCkKICAgICAgICBzZWxmLmRyb3BvdXQzID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYubm9ybTMgPSBubi5MYXllck5vcm0oZF9tb2RlbCkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIHg6IFRlbnNvciwKICAgICAgICBlbmNvZGVyX291dHB1dDogVGVuc29yLAogICAgICAgIHRndF9tYXNrOiBUZW5zb3IsCiAgICAgICAgc3JjX21hc2s6IFRlbnNvciwKICAgICkgLT4gVGVuc29yOgogICAgICAgIHNlbGZfYXR0bl9vdXQgPSBzZWxmLnNlbGZfYXR0bih4LCB4LCB4LCB0Z3RfbWFzaykKICAgICAgICB4ID0gc2VsZi5ub3JtMSh4ICsgc2VsZi5kcm9wb3V0MShzZWxmX2F0dG5fb3V0KSkKCiAgICAgICAgY3Jvc3NfYXR0bl9vdXQgPSBzZWxmLmNyb3NzX2F0dG4oeCwgZW5jb2Rlcl9vdXRwdXQsIGVuY29kZXJfb3V0cHV0LCBzcmNfbWFzaykKICAgICAgICB4ID0gc2VsZi5ub3JtMih4ICsgc2VsZi5kcm9wb3V0Mihjcm9zc19hdHRuX291dCkpCgogICAgICAgIGZmX291dCA9IHNlbGYuZmVlZF9mb3J3YXJkKHgpCiAgICAgICAgeCA9IHNlbGYubm9ybTMoeCArIHNlbGYuZHJvcG91dDMoZmZfb3V0KSkKICAgICAgICByZXR1cm4geAoKCmNsYXNzIERlY29kZXIobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHZvY2FiX3NpemU6IGludCwKICAgICAgICBkX21vZGVsOiBpbnQgPSBEX01PREVMLAogICAgICAgIG5faGVhZHM6IGludCA9IE5fSEVBRFMsCiAgICAgICAgZF9mZjogaW50ID0gRF9GRiwKICAgICAgICBuX2xheWVyczogaW50ID0gTl9ERUNPREVSX0xBWUVSUywKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IERST1BPVVQsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZW1iZWRkaW5ncyA9IEVtYmVkZGluZ3Modm9jYWJfc2l6ZSwgZF9tb2RlbCwgZHJvcG91dCkKICAgICAgICBzZWxmLmxheWVycyA9IG5uLk1vZHVsZUxpc3QoCiAgICAgICAgICAgIFtEZWNvZGVyQmxvY2soZF9tb2RlbCwgbl9oZWFkcywgZF9mZiwgZHJvcG91dCkgZm9yIF8gaW4gcmFuZ2Uobl9sYXllcnMpXQogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIHRndF9pZHM6IFRlbnNvciwKICAgICAgICBlbmNvZGVyX291dHB1dDogVGVuc29yLAogICAgICAgIHRndF9tYXNrOiBUZW5zb3IsCiAgICAgICAgc3JjX21hc2s6IFRlbnNvciwKICAgICkgLT4gVGVuc29yOgogICAgICAgIHggPSBzZWxmLmVtYmVkZGluZ3ModGd0X2lkcykKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnM6CiAgICAgICAgICAgIHggPSBsYXllcih4LCBlbmNvZGVyX291dHB1dCwgdGd0X21hc2ssIHNyY19tYXNrKQogICAgICAgIHJldHVybiB4Cg==", "src/model/transformer.py": "aW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0ICgKICAgIERfRkYsCiAgICBEX01PREVMLAogICAgRFJPUE9VVCwKICAgIE5fREVDT0RFUl9MQVlFUlMsCiAgICBOX0VOQ09ERVJfTEFZRVJTLAogICAgTl9IRUFEUywKICAgIFBBRF9JRCwKICAgIFRJRV9PVVRQVVRfUFJPSkVDVElPTiwKKQpmcm9tIHNyYy5tb2RlbC5kZWNvZGVyIGltcG9ydCBEZWNvZGVyCmZyb20gc3JjLm1vZGVsLmVuY29kZXIgaW1wb3J0IEVuY29kZXIKZnJvbSBzcmMubW9kZWwubWFza3MgaW1wb3J0ICgKICAgIG1ha2VfY3Jvc3NfYXR0bl9tYXNrLAogICAgbWFrZV9kZWNvZGVyX3NlbGZfYXR0bl9tYXNrLAogICAgbWFrZV9wYWRkaW5nX21hc2ssCikKCgpjbGFzcyBTZXEyU2VxVHJhbnNmb3JtZXIobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHNyY192b2NhYl9zaXplOiBpbnQsCiAgICAgICAgdGd0X3ZvY2FiX3NpemU6IGludCwKICAgICAgICBkX21vZGVsOiBpbnQgPSBEX01PREVMLAogICAgICAgIG5faGVhZHM6IGludCA9IE5fSEVBRFMsCiAgICAgICAgZF9mZjogaW50ID0gRF9GRiwKICAgICAgICBuX2VuY29kZXJfbGF5ZXJzOiBpbnQgPSBOX0VOQ09ERVJfTEFZRVJTLAogICAgICAgIG5fZGVjb2Rlcl9sYXllcnM6IGludCA9IE5fREVDT0RFUl9MQVlFUlMsCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSBEUk9QT1VULAogICAgICAgIHBhZF9pZDogaW50ID0gUEFEX0lELAogICAgICAgIHRpZV9vdXRwdXRfcHJvamVjdGlvbjogYm9vbCA9IFRJRV9PVVRQVVRfUFJPSkVDVElPTiwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5wYWRfaWQgPSBwYWRfaWQKCiAgICAgICAgc2VsZi5lbmNvZGVyID0gRW5jb2RlcihzcmNfdm9jYWJfc2l6ZSwgZF9tb2RlbCwgbl9oZWFkcywgZF9mZiwgbl9lbmNvZGVyX2xheWVycywgZHJvcG91dCkKICAgICAgICBzZWxmLmRlY29kZXIgPSBEZWNvZGVyKHRndF92b2NhYl9zaXplLCBkX21vZGVsLCBuX2hlYWRzLCBkX2ZmLCBuX2RlY29kZXJfbGF5ZXJzLCBkcm9wb3V0KQogICAgICAgIHNlbGYub3V0cHV0X3Byb2plY3Rpb24gPSBubi5MaW5lYXIoZF9tb2RlbCwgdGd0X3ZvY2FiX3NpemUpCgogICAgICAgIGlmIHRpZV9vdXRwdXRfcHJvamVjdGlvbjoKICAgICAgICAgICAgc2VsZi5vdXRwdXRfcHJvamVjdGlvbi53ZWlnaHQgPSBzZWxmLmRlY29kZXIuZW1iZWRkaW5ncy50b2tlbl9lbWJlZGRpbmcuZW1iZWRkaW5nLndlaWdodAoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHNyY19pZHM6IFRlbnNvciwgdGd0X2lkczogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgc3JjX21hc2sgPSBtYWtlX3BhZGRpbmdfbWFzayhzcmNfaWRzLCBzZWxmLnBhZF9pZCkKICAgICAgICB0Z3RfbWFzayA9IG1ha2VfZGVjb2Rlcl9zZWxmX2F0dG5fbWFzayh0Z3RfaWRzLCBzZWxmLnBhZF9pZCkKICAgICAgICBjcm9zc19tYXNrID0gbWFrZV9jcm9zc19hdHRuX21hc2soc3JjX2lkcywgc2VsZi5wYWRfaWQpCgogICAgICAgIGVuY29kZXJfb3V0cHV0ID0gc2VsZi5lbmNvZGVyKHNyY19pZHMsIHNyY19tYXNrKQogICAgICAgIGRlY29kZXJfb3V0cHV0ID0gc2VsZi5kZWNvZGVyKHRndF9pZHMsIGVuY29kZXJfb3V0cHV0LCB0Z3RfbWFzaywgY3Jvc3NfbWFzaykKICAgICAgICByZXR1cm4gc2VsZi5vdXRwdXRfcHJvamVjdGlvbihkZWNvZGVyX291dHB1dCkK", "src/training/lr_schedule.py": "ZnJvbSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIgaW1wb3J0IExhbWJkYUxSCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgRF9NT0RFTCwgV0FSTVVQX1NURVBTCgoKZGVmIG5vYW1fbHJfbGFtYmRhKHN0ZXA6IGludCkgLT4gZmxvYXQ6CiAgICBzdGVwID0gbWF4KHN0ZXAsIDEpCiAgICByZXR1cm4gKERfTU9ERUwgKiogLTAuNSkgKiBtaW4oc3RlcCAqKiAtMC41LCBzdGVwICogKFdBUk1VUF9TVEVQUyAqKiAtMS41KSkKCgpkZWYgYnVpbGRfbm9hbV9zY2hlZHVsZXIob3B0aW1pemVyKToKICAgIHJldHVybiBMYW1iZGFMUihvcHRpbWl6ZXIsIGxyX2xhbWJkYT1ub2FtX2xyX2xhbWJkYSkK", "src/training/checkpoint.py": "ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKCmZyb20gY29uZmlncy5iYXNlIGltcG9ydCBDSEVDS1BPSU5UX0RJUgoKCmRlZiBzYXZlX2NoZWNrcG9pbnQobW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBzdGVwOiBpbnQsIGVwb2NoOiBpbnQsIHZhbF9sb3NzOiBmbG9hdCwgbmFtZTogc3RyKToKICAgIENIRUNLUE9JTlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGggPSBQYXRoKENIRUNLUE9JTlRfRElSKSAvIGYie25hbWV9LnB0IgogICAgdG9yY2guc2F2ZSgKICAgICAgICB7CiAgICAgICAgICAgICJtb2RlbF9zdGF0ZSI6IG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZSI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICJzY2hlZHVsZXJfc3RhdGUiOiBzY2hlZHVsZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAic3RlcCI6IHN0ZXAsCiAgICAgICAgICAgICJlcG9jaCI6IGVwb2NoLAogICAgICAgICAgICAidmFsX2xvc3MiOiB2YWxfbG9zcywKICAgICAgICB9LAogICAgICAgIHBhdGgsCiAgICApCiAgICByZXR1cm4gcGF0aAoKCmRlZiBsb2FkX2NoZWNrcG9pbnQocGF0aCwgbW9kZWwsIG9wdGltaXplcj1Ob25lLCBzY2hlZHVsZXI9Tm9uZSwgbWFwX2xvY2F0aW9uPSJjcHUiKToKICAgIGNrcHQgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj1tYXBfbG9jYXRpb24pCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2twdFsibW9kZWxfc3RhdGUiXSkKICAgIGlmIG9wdGltaXplciBpcyBub3QgTm9uZSBhbmQgIm9wdGltaXplcl9zdGF0ZSIgaW4gY2twdDoKICAgICAgICBvcHRpbWl6ZXIubG9hZF9zdGF0ZV9kaWN0KGNrcHRbIm9wdGltaXplcl9zdGF0ZSJdKQogICAgaWYgc2NoZWR1bGVyIGlzIG5vdCBOb25lIGFuZCAic2NoZWR1bGVyX3N0YXRlIiBpbiBja3B0OgogICAgICAgIHNjaGVkdWxlci5sb2FkX3N0YXRlX2RpY3QoY2twdFsic2NoZWR1bGVyX3N0YXRlIl0pCiAgICByZXR1cm4gY2twdAo=", "src/training/train.py": "aW1wb3J0IHRpbWUKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIsIFN1YnNldAoKZnJvbSBjb25maWdzLmJhc2UgaW1wb3J0IEFEQU1fQkVUQVMsIEFEQU1fRVBTLCBHUkFEX0NMSVBfTk9STSwgUEFEX0lECmZyb20gc3JjLmRhdGEuZGF0YXNldCBpbXBvcnQgVHJhbnNsYXRpb25EYXRhc2V0LCBjb2xsYXRlX2ZuCmZyb20gc3JjLm1vZGVsLnRyYW5zZm9ybWVyIGltcG9ydCBTZXEyU2VxVHJhbnNmb3JtZXIKZnJvbSBzcmMudG9rZW5pemF0aW9uLnRva2VuaXplcl91dGlscyBpbXBvcnQgbG9hZF90b2tlbml6ZXIKZnJvbSBzcmMudG9rZW5pemF0aW9uLnRyYWluX3Rva2VuaXplciBpbXBvcnQgRU5fVE9LRU5JWkVSX1BBVEgsIE9SX1RPS0VOSVpFUl9QQVRICmZyb20gc3JjLnRyYWluaW5nLmNoZWNrcG9pbnQgaW1wb3J0IHNhdmVfY2hlY2twb2ludApmcm9tIHNyYy50cmFpbmluZy5scl9zY2hlZHVsZSBpbXBvcnQgYnVpbGRfbm9hbV9zY2hlZHVsZXIKCgpkZWYgYnVpbGRfbW9kZWwoKToKICAgIGVuX3ZvY2FiX3NpemUgPSBsb2FkX3Rva2VuaXplcihFTl9UT0tFTklaRVJfUEFUSCkuZ2V0X3ZvY2FiX3NpemUoKQogICAgb3Jfdm9jYWJfc2l6ZSA9IGxvYWRfdG9rZW5pemVyKE9SX1RPS0VOSVpFUl9QQVRIKS5nZXRfdm9jYWJfc2l6ZSgpCiAgICByZXR1cm4gU2VxMlNlcVRyYW5zZm9ybWVyKGVuX3ZvY2FiX3NpemUsIG9yX3ZvY2FiX3NpemUpCgoKZGVmIHJ1bl9lcG9jaChtb2RlbCwgbG9hZGVyLCBsb3NzX2ZuLCBvcHRpbWl6ZXI9Tm9uZSwgc2NoZWR1bGVyPU5vbmUsIGRldmljZT0iY3B1Iik6CiAgICBpc190cmFpbiA9IG9wdGltaXplciBpcyBub3QgTm9uZQogICAgbW9kZWwudHJhaW4oaXNfdHJhaW4pCiAgICB0b3RhbF9sb3NzLCB0b3RhbF90b2tlbnMgPSAwLjAsIDAKCiAgICBmb3Igc3JjX2lkcywgX3NyY19wYWRfbWFzaywgdGd0X2lkcywgX3RndF9wYWRfbWFzayBpbiBsb2FkZXI6CiAgICAgICAgc3JjX2lkcywgdGd0X2lkcyA9IHNyY19pZHMudG8oZGV2aWNlKSwgdGd0X2lkcy50byhkZXZpY2UpCiAgICAgICAgZGVjb2Rlcl9pbnB1dCA9IHRndF9pZHNbOiwgOi0xXQogICAgICAgIGRlY29kZXJfdGFyZ2V0ID0gdGd0X2lkc1s6LCAxOl0KCiAgICAgICAgd2l0aCB0b3JjaC5zZXRfZ3JhZF9lbmFibGVkKGlzX3RyYWluKToKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoc3JjX2lkcywgZGVjb2Rlcl9pbnB1dCkKICAgICAgICAgICAgbG9zcyA9IGxvc3NfZm4obG9naXRzLnJlc2hhcGUoLTEsIGxvZ2l0cy5zaXplKC0xKSksIGRlY29kZXJfdGFyZ2V0LnJlc2hhcGUoLTEpKQoKICAgICAgICBpZiBpc190cmFpbjoKICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpCiAgICAgICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICBubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVycygpLCBHUkFEX0NMSVBfTk9STSkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCgogICAgICAgIG5fdG9rZW5zID0gKGRlY29kZXJfdGFyZ2V0ICE9IFBBRF9JRCkuc3VtKCkuaXRlbSgpCiAgICAgICAgdG90YWxfbG9zcyArPSBsb3NzLml0ZW0oKSAqIG5fdG9rZW5zCiAgICAgICAgdG90YWxfdG9rZW5zICs9IG5fdG9rZW5zCgogICAgcmV0dXJuIHRvdGFsX2xvc3MgLyBtYXgodG90YWxfdG9rZW5zLCAxKQoKCmRlZiB0cmFpbigKICAgIHRyYWluX3BhdGgsCiAgICB2YWxfcGF0aCwKICAgIGJhdGNoX3NpemUsCiAgICBudW1fZXBvY2hzLAogICAgY2hlY2twb2ludF9uYW1lLAogICAgZGV2aWNlPSJjcHUiLAogICAgbWF4X3RyYWluX2V4YW1wbGVzPU5vbmUsCiAgICBtYXhfdmFsX2V4YW1wbGVzPU5vbmUsCik6CiAgICBtb2RlbCA9IGJ1aWxkX21vZGVsKCkudG8oZGV2aWNlKQoKICAgIHRyYWluX2RzID0gVHJhbnNsYXRpb25EYXRhc2V0KHRyYWluX3BhdGgpCiAgICB2YWxfZHMgPSBUcmFuc2xhdGlvbkRhdGFzZXQodmFsX3BhdGgpCiAgICBpZiBtYXhfdHJhaW5fZXhhbXBsZXMgaXMgbm90IE5vbmU6CiAgICAgICAgdHJhaW5fZHMgPSBTdWJzZXQodHJhaW5fZHMsIHJhbmdlKG1pbihtYXhfdHJhaW5fZXhhbXBsZXMsIGxlbih0cmFpbl9kcykpKSkKICAgIGlmIG1heF92YWxfZXhhbXBsZXMgaXMgbm90IE5vbmU6CiAgICAgICAgdmFsX2RzID0gU3Vic2V0KHZhbF9kcywgcmFuZ2UobWluKG1heF92YWxfZXhhbXBsZXMsIGxlbih2YWxfZHMpKSkpCgogICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPVRydWUsIGNvbGxhdGVfZm49Y29sbGF0ZV9mbikKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9kcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLCBjb2xsYXRlX2ZuPWNvbGxhdGVfZm4pCgogICAgIyBiYXNlIGxyPTEuMDogdGhlIE5vYW0gc2NoZWR1bGUgY29tcHV0ZXMgdGhlIGFjdHVhbCBsciBhcyBhIG11bHRpcGxpZXIsIGFwcGxpZWQgYnkgTGFtYmRhTFIKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj0xLjAsIGJldGFzPUFEQU1fQkVUQVMsIGVwcz1BREFNX0VQUykKICAgIHNjaGVkdWxlciA9IGJ1aWxkX25vYW1fc2NoZWR1bGVyKG9wdGltaXplcikKICAgIGxvc3NfZm4gPSBubi5Dcm9zc0VudHJvcHlMb3NzKGlnbm9yZV9pbmRleD1QQURfSUQpCgogICAgYmVzdF92YWxfbG9zcyA9IGZsb2F0KCJpbmYiKQogICAgaGlzdG9yeSA9IFtdCiAgICBnbG9iYWxfc3RlcCA9IDAKICAgIGZvciBlcG9jaCBpbiByYW5nZSgxLCBudW1fZXBvY2hzICsgMSk6CiAgICAgICAgc3RhcnQgPSB0aW1lLnRpbWUoKQogICAgICAgIHRyYWluX2xvc3MgPSBydW5fZXBvY2gobW9kZWwsIHRyYWluX2xvYWRlciwgbG9zc19mbiwgb3B0aW1pemVyLCBzY2hlZHVsZXIsIGRldmljZSkKICAgICAgICB2YWxfbG9zcyA9IHJ1bl9lcG9jaChtb2RlbCwgdmFsX2xvYWRlciwgbG9zc19mbiwgTm9uZSwgTm9uZSwgZGV2aWNlKQogICAgICAgIGdsb2JhbF9zdGVwICs9IGxlbih0cmFpbl9sb2FkZXIpCiAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gc3RhcnQKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJlcG9jaCB7ZXBvY2h9L3tudW1fZXBvY2hzfSB0cmFpbl9sb3NzPXt0cmFpbl9sb3NzOi40Zn0gIgogICAgICAgICAgICBmInZhbF9sb3NzPXt2YWxfbG9zczouNGZ9ICh7ZWxhcHNlZDouMWZ9cykiCiAgICAgICAgKQogICAgICAgIGhpc3RvcnkuYXBwZW5kKHsiZXBvY2giOiBlcG9jaCwgInRyYWluX2xvc3MiOiB0cmFpbl9sb3NzLCAidmFsX2xvc3MiOiB2YWxfbG9zc30pCgogICAgICAgIHNhdmVfY2hlY2twb2ludChtb2RlbCwgb3B0aW1pemVyLCBzY2hlZHVsZXIsIGdsb2JhbF9zdGVwLCBlcG9jaCwgdmFsX2xvc3MsIGYie2NoZWNrcG9pbnRfbmFtZX1fbGFzdCIpCiAgICAgICAgaWYgdmFsX2xvc3MgPCBiZXN0X3ZhbF9sb3NzOgogICAgICAgICAgICBiZXN0X3ZhbF9sb3NzID0gdmFsX2xvc3MKICAgICAgICAgICAgc2F2ZV9jaGVja3BvaW50KG1vZGVsLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgZ2xvYmFsX3N0ZXAsIGVwb2NoLCB2YWxfbG9zcywgZiJ7Y2hlY2twb2ludF9uYW1lfV9iZXN0IikKCiAgICByZXR1cm4gbW9kZWwsIGhpc3RvcnkK", "src/inference/greedy_decode.py": "aW1wb3J0IHRvcmNoCgpmcm9tIGNvbmZpZ3MuYmFzZSBpbXBvcnQgRU9TX0lELCBHUkVFRFlfTUFYX0RFQ09ERV9MRU4sIFNPU19JRAoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGdyZWVkeV9kZWNvZGUobW9kZWwsIHNyY19pZHM6IHRvcmNoLlRlbnNvciwgbWF4X2xlbjogaW50ID0gR1JFRURZX01BWF9ERUNPREVfTEVOKSAtPiBsaXN0W2ludF06CiAgICBkZXZpY2UgPSBzcmNfaWRzLmRldmljZQogICAgd2FzX3RyYWluaW5nID0gbW9kZWwudHJhaW5pbmcKICAgIG1vZGVsLmV2YWwoKQoKICAgIHRndF9pZHMgPSB0b3JjaC50ZW5zb3IoW1tTT1NfSURdXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPWRldmljZSkKICAgIGZvciBfIGluIHJhbmdlKG1heF9sZW4gLSAxKToKICAgICAgICBsb2dpdHMgPSBtb2RlbChzcmNfaWRzLCB0Z3RfaWRzKQogICAgICAgIG5leHRfaWQgPSBsb2dpdHNbOiwgLTEsIDpdLmFyZ21heChkaW09LTEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICB0Z3RfaWRzID0gdG9yY2guY2F0KFt0Z3RfaWRzLCBuZXh0X2lkXSwgZGltPTEpCiAgICAgICAgaWYgbmV4dF9pZC5pdGVtKCkgPT0gRU9TX0lEOgogICAgICAgICAgICBicmVhawoKICAgIG1vZGVsLnRyYWluKHdhc190cmFpbmluZykKICAgIHJldHVybiB0Z3RfaWRzWzBdLnRvbGlzdCgpCg=="}

for rel_path, b64_src in _FILES_B64.items():
    dest = REPO_ROOT / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(base64.b64decode(b64_src))

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"materialized {len(_FILES_B64)} files under {REPO_ROOT}")


## Data
Load the pre-processed dataset from the attached Kaggle dataset input if
present; otherwise fall back to running the download/clean/tokenize/split
pipeline inline (requires internet, enabled on this kernel).


In [ ]:
import shutil
from pathlib import Path

KAGGLE_INPUT_CANDIDATES = list(Path("/kaggle/input").glob("*/processed"))
DATA_PROCESSED_DIR = Path("/kaggle/working/data/processed")
TOKENIZER_DIR = Path("/kaggle/working/tokenizers")

if KAGGLE_INPUT_CANDIDATES:
    src_dir = KAGGLE_INPUT_CANDIDATES[0]
    DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    for f in src_dir.glob("*.parquet"):
        shutil.copy(f, DATA_PROCESSED_DIR / f.name)
    input_root = src_dir.parent
    tok_src = input_root / "tokenizers"
    if tok_src.exists():
        TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
        for f in tok_src.glob("*.json"):
            shutil.copy(f, TOKENIZER_DIR / f.name)
    print(f"copied processed data + tokenizers from {src_dir.parent}")
else:
    print("no Kaggle dataset input found, running the pipeline inline")
    from src.data.download import main as download_main
    from src.tokenization.train_tokenizer import main as train_tokenizer_main
    from src.data.split import main as split_main

    download_main()
    train_tokenizer_main()
    split_main()


## Train (Kaggle GPU profile: full dataset, larger batch size, full epoch count)


In [ ]:
import torch
from configs.base import (
    BATCH_SIZE_KAGGLE,
    DATA_PROCESSED_DIR,
    NUM_EPOCHS_KAGGLE,
)
from src.training.train import train

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"training on device: {device}")

model, history = train(
    train_path=DATA_PROCESSED_DIR / "train.parquet",
    val_path=DATA_PROCESSED_DIR / "val.parquet",
    batch_size=BATCH_SIZE_KAGGLE,
    num_epochs=NUM_EPOCHS_KAGGLE,
    checkpoint_name="kaggle_run",
    device=device,
)


## Done
Checkpoints are written under `/kaggle/working/checkpoints/` and will appear
in this kernel's Output tab for download once the run finishes.
